<a href="https://colab.research.google.com/github/balajiduddukuri/Langchain_practice/blob/Ultimate-Content-Repurposer/langraph_apr_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [182]:
!pip install -q openai langchain chromadb faiss-cpu pypdf tiktoken docarray langchain-community tavily-python langchain-experimental

In [183]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get("api_key")

print("Key loaded successfully")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    api_key=OPENAI_API_KEY
)

Key loaded successfully


In [ ]:
#NODE

In [ ]:
#EDGE

In [ ]:
#STEP

# LangGrapgh
# AI Agents
# Multi-step wrkflows
# RAG pipelines
# Autonomous Systems

In [184]:
!pip install -q langgraph langchain langchain-openai

In [185]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

In [186]:
class MyState(TypedDict):
  topic: str
  explanation: str
  summary: str
  quiz: str

In [187]:
llm = ChatOpenAI(model = "gpt-4o-mini",api_key=OPENAI_API_KEY, temperature = 0.7)

In [188]:
#NODE1:explain

def explain_node(state: MyState):
  response = llm.invoke(f"Explain {state["topic"]} in simple terms")
  return {
      "explanation": response.content
  }

In [189]:
#Node2:summarize
def summarize_node(state: MyState):
  response = llm.invoke(f"Summarize this in 2 lines:\n{state["explanation"]}")
  return {
      "summary": response.content
  }

In [190]:
#Node3:Quiz
def quiz_node(state: MyState):
  response = llm.invoke(f"Quiz me on this with 2 questions:\n{state["explanation"]}")
  return  {
      "quiz": response.content
  }

In [191]:
graph = StateGraph(MyState)

graph.add_node("explain", explain_node)
graph.add_node("summarize", summarize_node)
graph.add_node("quiz", quiz_node)

graph.set_entry_point("explain")
graph.add_edge("explain", "summarize")
graph.add_edge("summarize", "quiz")
graph.add_edge("quiz", END)

In [192]:
app = graph.compile()

In [193]:
result = app.invoke({
    "topic": "Transformers"
})

print("\nExplanation:\n", result["explanation"])
print("\nSummary:\n", result["summary"])
print("\nquiz:\n", result["quiz"])


Explanation:
 Transformers are a type of model used in machine learning, especially for understanding and generating language. They work by processing words in a sentence all at once, rather than one by one in order. Here’s a simple breakdown:

1. **Words as Vectors**: Each word in a sentence is converted into a numerical format (called a vector) that captures its meaning.

2. **Attention Mechanism**: Transformers use something called "attention" to determine which words in a sentence are important for understanding other words. For example, in the sentence "The cat sat on the mat," the model can focus on the word "cat" to understand more about "sat."

3. **Layers**: A transformer is made up of many layers that process the information. Each layer refines the understanding of the words and their relationships.

4. **Parallel Processing**: Unlike older models that processed words one at a time, transformers can look at all the words in a sentence simultaneously. This makes them faster a

In [194]:
from IPython.display import Image, display

# Assuming 'app' is already compiled from previous steps
display(Image(app.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
#@haystack, @Autogen, @crewai

#conditional routing
#DEMO

In [195]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

In [196]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get("api_key")

In [197]:
class MyState(TypedDict):
  topic: str
  difficulty: str
  output: str

In [198]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0.7)

In [199]:
def classify_node(state: MyState):
  response = llm.invoke(
      f"Classify this topic as 'simple' or 'complex': {state['topic']}"
  )

  return {
      "difficulty": response.content.strip().lower()
  }

In [200]:
def simple_node(state: MyState):
  response = llm.invoke(
      f"Explain {state['topic']} in 2 simple lines"
  )

  return {
      "output": response.content
  }

In [201]:
def complex_node(state: MyState):
  response = llm.invoke(
      f"Explain {state['topic']} in detailed with example"
  )

  return {
      "output": response.content
  }

In [202]:
def route(state: MyState):
  if "simple" in state['difficulty']:
    return "simple"
  else:
    return "complex"

In [203]:
#Build graph

graph = StateGraph(MyState)
graph.add_node("classify", classify_node)
graph.add_node("simple", simple_node)
graph.add_node("complex", complex_node)


graph.set_entry_point("classify")

graph.add_conditional_edges(
    "classify",
    route,
    {
        "simple": "simple",
        "complex": "complex"
    }
)

graph.add_edge("simple", END)
graph.add_edge("complex", END)

In [204]:
app = graph.compile()

In [205]:
result = app.invoke({
    "topic": "Linear Regression"
})

print(result["output"])

Linear regression is a statistical method used to model the relationship between a dependent variable and one or more independent variables by fitting a linear equation to observed data. It aims to predict the value of the dependent variable based on the values of the independent variables.


In [206]:
result = app.invoke({
    "topic": "math involved in the Tansformer architecture"
})

print(result["output"])

The Transformer architecture, introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, has become the foundation for many state-of-the-art natural language processing (NLP) models. The math involved in the Transformer architecture mainly revolves around linear algebra, attention mechanisms, and positional encodings. Below, I'll break down the key components of the Transformer architecture with a detailed explanation and examples.

### 1. Overview of the Transformer Architecture

The Transformer consists of an Encoder-Decoder structure:

- **Encoder**: Processes the input sequence and generates a set of continuous representations.
- **Decoder**: Processes the representations from the encoder and generates the output sequence.

Each of these components consists of multiple layers, and within each layer, there are two main sub-components: multi-head self-attention and feedforward neural networks.

### 2. Input Representation

Before inputting data into the model, we 

Input Topic | [Node: Classify] | Decision (Simple/ Complex) | |-> [Node: Simple] |-> [Node: Complex] | END

AI Agents -> Decide whether to call tool or LLM
RAG Systems -> If a question needs context -> retrieve
We could route it to different workflows based on the user's intent

How is the Tool Calling possible in LangGraph?

In [ ]:
def decision_node(state: MyState):
  response = llm.invoke(
      f"Decide if this needs a calculator or LLM: {state['UserQuery']}",
      f"Repond only with 'tool' or 'LLM'. "
  )
  return {
      "decision": response.content.strip().lower()
  }

##PRACTICE PROGRAM

#decision_node.py — your code, fixed & extended

In [207]:
#decision_node.py — your code, fixed & extended

from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

# ── 1. Define State ──────────────────────────────────────────
class MyState(TypedDict):
    UserQuery: str
    decision:  str           # "tool" | "llm"
    output:    str

# ── 2. Decision Node ─────────────────────────────────────────
def decision_node(state: MyState):
    # Fix: llm.invoke() takes a single argument (list of messages)
    response = llm.invoke([
        {"role": "system", "content": "Decide if this needs a calculator or LLM. Respond ONLY with 'tool' or 'llm'."},
        {"role": "user",   "content": state["UserQuery"]},
    ])
    decision = response.content.strip().lower()
    if decision not in ("tool", "llm"):
        decision = "llm"  # safe fallback
    return {"decision": decision}

# ── 3. Branch Nodes ──────────────────────────────────────────
def tool_node(state: MyState):
    # NOTE: 'calculator' tool is not defined here in this snippet.
    # For a full example, ensure a 'calculator' tool is available.
    # For now, we'll return a placeholder.
    return {"output": f"Tool called for: {state['UserQuery']}"}

def llm_node(state: MyState):
    response = llm.invoke([{"role": "user", "content": state["UserQuery"]}])
    return {"output": response.content}

# ── 4. Router Function (reads state, returns next node name) ──
def route_decision(state: MyState) -> Literal["tool", "llm"]:
    return state["decision"]   # LangGraph sends flow here

# ── 5. Build Graph ────────────────────────────────────────────
graph = StateGraph(MyState)
graph.add_node("decide",   decision_node)
graph.add_node("tool",     tool_node)
graph.add_node("llm",      llm_node)

graph.set_entry_point("decide")

# Conditional edge: run route_decision() after "decide"
graph.add_conditional_edges(
    "decide",
    route_decision,
    {"tool": "tool", "llm": "llm"}   # returned value → next node
)
graph.add_edge("tool", END)
graph.add_edge("llm",  END)

app = graph.compile()

In [208]:
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a poetry expert."),
    HumanMessage(content="Write a haiku about spring."),
]
response = llm.invoke(messages)   # 1 argument: a list of message objects
print(response.content)

Blossoms in the breeze,  
Whispers of a warm embrace,  
Life awakens soft.


In [209]:
messages = [
    {"role": "system",  "content": "You are a helpful assistant."},
    {"role": "user",     "content": "Hi, I'm Bob."},
    {"role": "assistant","content": "Hello, Bob! How can I help?"},
    {"role": "user",     "content": "What is 2+2?"},
]
response = llm.invoke(messages)
print(response.content)   # e.g. "2+2 is 4"

2 + 2 equals 4.


In [210]:
state = {"UserQuery": "What is the capital of France?"}
response = llm.invoke(
    [
        {"role": "system", "content": "Decide if this needs a calculator or LLM. Respond ONLY with 'tool' or 'llm'."},
        {"role": "user", "content": state["UserQuery"]},
    ]
)
decision = response.content.strip().lower()

In [211]:
print (decision)

llm


#bind_tools.py — idiomatic production pattern

In [212]:
#bind_tools.py — idiomatic production pattern
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI

# ── 1. Define tools with @tool decorator ─────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression))   # use numexpr in production

@tool
def weather_lookup(city: str) -> str:
    """Get current weather for a city."""
    return f"Weather in {city}: 28°C, sunny"   # stub

tools = [calculator, weather_lookup]

# ── 2. Bind tools to LLM ─────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)   # LLM knows which tools exist

# ── 3. State ─────────────────────────────────────────────────
class State(TypedDict):
    messages: Annotated[list, add_messages]  # auto-appends, no overwrite

# ── 4. Nodes ─────────────────────────────────────────────────
def agent_node(state: State):
    # LLM decides to call a tool or reply — returns AIMessage
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)   # prebuilt: auto-executes any tool call in last message

# ── 5. Graph with loop ───────────────────────────────────────
graph = StateGraph(State)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

# tools_condition: routes to "tools" if AIMessage has tool_calls, else END
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")  # loop back so LLM sees tool result

app = graph.compile()

# ── 6. Invoke ────────────────────────────────────────────────
result = app.invoke({"messages": [{"role": "user", "content": "What is 42 * 17?"}]})
print(result["messages"][-1].content)  # "The result is 714"

The result of \( 42 \times 17 \) is 714.


In [ ]:
result = app.invoke({"messages": [{"role": "user", "content": "What is FOUR MULTIPLIED BY SEVEN ?"}]})
print(result["messages"][-1].content)

In [214]:
print (result)

{'messages': [HumanMessage(content='What is 42 * 17?', additional_kwargs={}, response_metadata={}, id='7a702589-43c7-4823-9189-a2b661098985'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 70, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQxnGNJ5VEJA1pvdafTeILgTqftKD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d593e-469d-7101-a378-307870e3acc5-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '42 * 17'}, 'id': 'call_8vymoGr7D7ncgJxuEC9LiYeT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens'

#rag_routing.py — conditional RAG retrieval

In [216]:
#rag_routing.py — conditional RAG retrieval
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal, Optional

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

class RAGState(TypedDict):
    question:  str
    context:   Optional[str]
    answer:    str
    needs_rag: bool

# Define some dummy product documentation for demonstration
docs = [
    Document(page_content="Our Basic plan costs $10/month and includes up to 1000 API calls."),
    Document(page_content="The Pro plan is $50/month, offering 10,000 API calls and priority support."),
    Document(page_content="Enterprise plans are customized and include unlimited API calls, dedicated support, and on-premise deployment options."),
    Document(page_content="To configure SSO, navigate to 'Settings' -> 'Security' and enable 'Single Sign-On'. You will need your Identity Provider's metadata URL."),
    Document(page_content="The capital of France is Paris. It is a beautiful city known for its art and culture."),
    Document(page_content="What is 2+2? The answer is 4."),
]

# Initialize OpenAI Embeddings (assuming OPENAI_API_KEY is already defined)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# Create a FAISS vector store from the documents
vectorstore = FAISS.from_documents(docs, embeddings)

# ── Router: does this question need external context? ─────────
def classify_question(state: RAGState):
    response = llm.invoke([
        {"role": "system", "content": (
            "Does the question need product docs to answer? "
            "Reply ONLY with 'rag' or 'direct'."
        )},
        {"role": "user", "content": state["question"]},
    ])
    needs = response.content.strip().lower() == "rag"
    return {"needs_rag": needs}

def route_rag(state: RAGState) -> Literal["retrieve", "answer_direct"]:
    return "retrieve" if state["needs_rag"] else "answer_direct"

# ── Retrieval Node ────────────────────────────────────────────
def retrieve(state: RAGState):
    docs = vectorstore.similarity_search(state["question"], k=4)
    context = "\n\n".join(d.page_content for d in docs)
    return {"context": context}

# ── Answer Nodes ─────────────────────────────────────────────
def answer_with_context(state: RAGState):
    response = llm.invoke([
        {"role": "system", "content": "Answer using the context below."},
        {"role": "user", "content": f"Context:\n{state['context']}\n\nQ: {state['question']}"},
    ])
    return {"answer": response.content}

def answer_direct(state: RAGState):
    response = llm.invoke([{"role": "user", "content": state["question"]}])
    return {"answer": response.content}

# ── Graph ─────────────────────────────────────────────────────
graph = StateGraph(RAGState)
graph.add_node("classify",      classify_question)
graph.add_node("retrieve",      retrieve)
graph.add_node("answer_rag",    answer_with_context)
graph.add_node("answer_direct", answer_direct)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_rag, {
    "retrieve":      "retrieve",
    "answer_direct": "answer_direct",
})
graph.add_edge("retrieve", "answer_rag")
graph.add_edge("answer_rag",    END)
graph.add_edge("answer_direct", END)

rag_app = graph.compile()

In [217]:
result = rag_app.invoke({
    "question": "What is the capital of France?",
})
print(result["answer"])
# → goes classify → answer_direct (no retrieval)

The capital of France is Paris.


In [218]:
result = rag_app.invoke({
    "question": "What are the steps to configure SSO in our product?",
})
print(result["answer"])
# → classify → retrieve → answer_with_context

To configure Single Sign-On (SSO) in your product, follow these steps:

1. Navigate to 'Settings'.
2. Go to the 'Security' section.
3. Enable 'Single Sign-On'.
4. Enter your Identity Provider's metadata URL.

Make sure you have your Identity Provider's metadata URL ready before starting the configuration process.


In [219]:
for event in rag_app.stream(
    {"question": "Explain the pricing tiers according to the latest doc."},
    stream_mode="values"   # one state per node
):
    if "answer" in event:
        print("ANSWER:", event["answer"])

ANSWER: The pricing tiers are structured as follows:

1. **Basic Plan**: Priced at $10 per month, this plan allows for up to 1,000 API calls.
  
2. **Pro Plan**: This plan costs $50 per month and provides users with up to 10,000 API calls along with priority support.

3. **Enterprise Plans**: These are customized plans tailored to the specific needs of the organization. They include unlimited API calls, dedicated support, and options for on-premise deployment.

Each tier offers different levels of service and API call limits to accommodate various user needs and budgets.


#full_agent.py — multi-tool agent with loop & memory

In [225]:
!pip install -q openai langchain chromadb faiss-cpu pypdf tiktoken docarray langchain-community tavily-python langchain-experimental

In [223]:
!pip install -q langchain-community langchain-tavily tavily-python

In [228]:
#full_agent.py — multi-tool agent with loop & memory
'''
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Annotated, TypedDict

# Import necessary clients
from langchain_community.tools.tavily_research import TavilySearchResults
from langchain_tavily import TavilySearch, TavilyResearch
from langchain_experimental.utilities import PythonREPL

# Initialize clients
tavily_client = TavilyResearch(max_results=5)
python_repl = PythonREPL()

# ── Tools ────────────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    return tavily_client.invoke({"query": query})

@tool
def run_python(code: str) -> str:
    """Execute Python code and return output."""
    return python_repl.run(code)

@tool
def calculator(expr: str) -> str:
    """Evaluate a math expression."""
    import numexpr
    return str(numexpr.evaluate(expr).item())

tools = [search_web, run_python, calculator]
llm   = ChatOpenAI(model="gpt-4o", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# ── State (messages reducer merges lists automatically) ───────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# ── Nodes ─────────────────────────────────────────────────────
def call_model(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# ── Graph (ReAct loop: agent ↔ tools) ─────────────────────────
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    tools_condition,   # built-in: checks for tool_calls in AIMessage
    {"tools": "tools", END: END}
)
graph.add_edge("tools", "agent")   # loop: results go back to agent

# ── Compile with memory (conversation history per thread) ─────
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# ── Streaming invocation ──────────────────────────────────────
config = {"configurable": {"thread_id": "balu-session-1"}}
for event in app.stream(
    {"messages": [{"role": "user", "content": "What's 2^32 and who invented the transistor?"}]},
    config=config, stream_mode="values"
):
    last = event["messages"][-1]
    last.pretty_print()
    '''

from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from typing import Annotated, TypedDict


# ── NO TAVILY, no Tavily imports ─────────────────────────────────────
# Remove these:
# from langchain_community.tools.tavily_research import TavilySearchResults
# from langchain_tavily import TavilySearch, TavilyResearch
# from langchain_experimental.utilities import PythonREPL


# Use PythonREPL if you already have it; otherwise mock or skip
try:
    from langchain_experimental.utilities import PythonREPL
    python_repl = PythonREPL()
except ImportError:
    python_repl = None


# ── Tools (mock search) ─────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """Mock web search; simulates real search without API key."""
    return f"Mock result for query '{query}': 2026 population of Tokyo ≈ 37 million. Transistor invented by John Bardeen, Walter Brattain, and William Shockley in 1947."


@tool
def run_python(code: str) -> str:
    """Execute Python code (if python_repl is available); otherwise mock."""
    if python_repl is None:
        return f"Mock Python result for {code!r}: 123.45"
    return python_repl.run(code)


@tool
def calculator(expr: str) -> str:
    """Evaluate a math expression."""
    import numexpr
    return str(numexpr.evaluate(expr).item())


tools = [search_web, run_python, calculator]
llm   = ChatOpenAI(model="gpt-4o", temperature=0)
llm_with_tools = llm.bind_tools(tools)


# ── State (messages reducer merges lists automatically) ───────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# ── Nodes ─────────────────────────────────────────────────────
def call_model(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ── Graph (ReAct loop: agent ↔ tools) ─────────────────────────
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    tools_condition,   # built-in: checks for tool_calls in AIMessage
    {"tools": "tools", END: END}
)
graph.add_edge("tools", "agent")   # loop: results go back to agent


# ── Compile with memory (conversation history per thread) ─────
memory = MemorySaver()
app = graph.compile(checkpointer=memory)


# ── Streaming invocation ──────────────────────────────────────
config = {"configurable": {"thread_id": "balu-session-1"}}
for event in app.stream(
    {"messages": [{"role": "user", "content": "What's 2^32 and who invented the transistor?"}]},
    config=config,
    stream_mode="values"
):
    last = event["messages"][-1]
    last.pretty_print()

================================ Human Message =================================

What's 2^32 and who invented the transistor?
================================== Ai Message ==================================
Tool Calls:
  calculator (call_7bARrxa1xM8L4R6lNqsTN9Sm)
 Call ID: call_7bARrxa1xM8L4R6lNqsTN9Sm
  Args:
    expr: 2^32
  search_web (call_zlLUO1g7l6A1EnpMIyFDlMVg)
 Call ID: call_zlLUO1g7l6A1EnpMIyFDlMVg
  Args:
    query: who invented the transistor
================================= Tool Message =================================
Name: search_web

Mock result for query 'who invented the transistor': 2026 population of Tokyo ≈ 37 million. Transistor invented by John Bardeen, Walter Brattain, and William Shockley in 1947.
================================== Ai Message ==================================

The value of \(2^{32}\) is 4,294,967,296.

The transistor was invented by John Bardeen, Walter Brattain, and William Shockley in 1947.


In [230]:
# ── Invoke with three sessions (balu-session-1, 2, 3) ─────────────
def run_test(thread_id: str, queries: list[str]):
    print(f"\n=== 🧪 {thread_id} ===")
    for q in queries:
        print(f"\nUser: {q}")
        result = app.invoke(
            {"messages": [{"role": "user", "content": q}]},
            {"configurable": {"thread_id": thread_id}},  # memory per thread
        )
        last = result["messages"][-1]
        last.pretty_print()

In [231]:
# Session 1: basic math + search
run_test(
    thread_id="balu-session-1",
    queries=[
        "Compute 2^32",
        "Who invented the transistor?",
    ]
)

# Session 2: chaining math + info
run_test(
    thread_id="balu-session-2",
    queries=[
        "What is 1024 * 1024?",
        "Now, explain what a transistor is in simple terms.",
    ]
)

# Session 3: mixed tools, memory‑aware
run_test(
    thread_id="balu-session-3",
    queries=[
        "What is the birth year of Alan Turing?",
        "Now, tell me 3 interesting facts about his life.",
    ]
)


=== 🧪 balu-session-1 ===

User: Compute 2^32
================================== Ai Message ==================================

The value of \(2^{32}\) is 34.

User: Who invented the transistor?
================================== Ai Message ==================================

The transistor was invented by John Bardeen, Walter Brattain, and William Shockley in 1947.

=== 🧪 balu-session-2 ===

User: What is 1024 * 1024?
================================== Ai Message ==================================

1024 multiplied by 1024 equals 1,048,576.

User: Now, explain what a transistor is in simple terms.
================================== Ai Message ==================================

A transistor is a small electronic device that can act as a switch or an amplifier for electrical signals. It's made of semiconductor material, usually silicon, and has three parts called the emitter, base, and collector. In simple terms, a transistor can turn a current on or off, or it can increase the strength

In [221]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

# ── Tools stub (you already defined these in full_agent.py) ────────────
# (replace with your actual tavily_client / python_repl if you prefer real ones)
def search_web(query: str) -> str:
    return f"Mock search result for '{query}': Tokyo population is about 37 million."

def run_python(code: str) -> str:
    return f"Mock result for {code!r}: 123.45"

def calculator(expr: str) -> str:
    # in prod use numexpr; this is for demo only
    return str(eval(expr))

tools = [search_web, run_python, calculator]

# ── Re‑import your compiled app (or paste full_agent.py here) ─────────
# Suppose your app is already compiled as:
# app = graph.compile(checkpointer=MemorySaver())

# ── 1. SINGLE TEST INVOKE (simple message) ─────────────────────────────
print("=== 1. Simple calculator test ===")
result = app.invoke(
    {"messages": [HumanMessage(content="What is 42 * 17?")]},
    {"configurable": {"thread_id": "balu-session-1"}},
)
msg = result["messages"][-1]
print("LLM →", msg.content)
print("Tool calls (if any):", getattr(msg, "tool_calls", []))
print()

# ── 2. TEST TOOL INVOCATION (multiple turns) ──────────────────────────
print("=== 2. Tool + LLM loop (ReAct) ===")
for event in app.stream(
    {"messages": [HumanMessage(content="Search for the population of Tokyo and calculate 10% of it")]},
    {"configurable": {"thread_id": "balu-session-2"}},
    stream_mode="values",
):
    last = event["messages"][-1]
    print(last)
    if last.content:
        print("→ LLM says:", last.content)
    if getattr(last, "tool_calls", []):
        print("→ tools:", [(tc["name"], tc["args"]) for tc in last.tool_calls])
    print()

# ── 3. CONVERSATION TEST WITH MEMORY (same thread) ─────────────────────
print("=== 3. Multi‑turn with memory (same thread_id) ===")
for query in [
    "What year was the transistor invented?",
    "Now summarize the last two facts in one sentence.",
]:
    print("\nUser:", query)
    result = app.invoke(
        {"messages": [HumanMessage(content=query)]},
        {"configurable": {"thread_id": "balu-session-2"}},  # reuse thread
    )
    msg = result["messages"][-1]
    print("Assistant:", msg.content)

=== 1. Simple calculator test ===
LLM → The result of \( 42 \times 17 \) is 714.
Tool calls (if any): []

=== 2. Tool + LLM loop (ReAct) ===
content='Search for the population of Tokyo and calculate 10% of it' additional_kwargs={} response_metadata={} id='c0113aa7-07f9-4307-9ccf-d99b2702cab0'
→ LLM says: Search for the population of Tokyo and calculate 10% of it

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 75, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQxpA0ATGhZLLVU2e019areHCwBYl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d5940-1093-7952-8c08-3b02cd656910